# 11 — SQLite Persistence

**Related guide:** [12_SQLITE_PERSISTENCE.md](../docs/guides/12_SQLITE_PERSISTENCE.md)

**Related script:** [12_sqlite.py](../examples/12_sqlite.py)

This notebook follows the DX05 loop: **Concept → Construction → Inspection → Modification → Execution → Observation → Experiment**.

## Construction

Persist runtime metadata to SQLite, then reopen the runtime from the same settings.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from pyworkflowkit import RuntimeSettings, TaskHandle, WorkflowRuntime, task, workflow


@task
def persist_me() -> str:
    return "durable"


@workflow(id="notebooks.sqlite", version="1")
def demo() -> tuple[TaskHandle, ...]:
    return (persist_me,)

In [ ]:
with TemporaryDirectory() as directory:
    workspace = Path(directory)
    settings = RuntimeSettings.load(
        overrides={
            "runtime": {"workspace": workspace},
            "metadata": {
                "backend": "sqlite",
                "sqlite_path": "runtime.sqlite3",
                "sqlite_wal": False,
            },
        }
    )
    runtime = WorkflowRuntime(settings)
    runtime.register(persist_me.handler_ref, persist_me.handler)
    definition = demo.build()
    run = runtime.run(definition)
    reopened = WorkflowRuntime(settings)
    loaded = reopened.get_run(run.run_id)
    print(
        {
            "status": loaded.status.value,
            "database_exists": (workspace / "runtime.sqlite3").is_file(),
        }
    )

## Experiment

Switch back to the memory backend and explain why the second runtime cannot recover the first runtime's state.